# revT2V: compare reverse-time methods

All methods run on **one shared U-Net** (`revt2v.infer.MethodBank`) from the **same initial noise `z`**:

| Column | What it is |
|---|---|
| `teacher` | frozen ModelScope teacher, forward time, from `z` |
| `matched_target` | `reverse(teacher(latents = flip(z)))`, the per-sample reference a reverse generator started from `z` should reproduce |
| `attn_rotation` | trained self-rotation + LoRA (up-block temporal attn1) |
| `conv_oracle` | teacher with every temporal Conv3d kernel flipped in time; no training |
| `conv_student` | LoRA on temporal Conv3d (only if its checkpoint exists) |

**Caveat:** `conv_oracle` is an exact time-mirror of the teacher (`unet_flipped(z) == flip(unet(flip(z)))`), so it is equivalent to reversing the teacher's output. It's a **reference / upper bound**, not a learned method.

Metrics are per method vs `matched_target`: optical-flow direction cosine (+1 = same motion direction, -1 = opposite), motion ratio (flow magnitude vs target; **frozen** if < 0.3), mean pixel difference.

## 1. Setup

In [ ]:
# Change this to your GitHub username
GITHUB_USER = "silentlooop"

!git clone -q https://github.com/{GITHUB_USER}/revT2V.git /content/revT2V 2>/dev/null || git -C /content/revT2V pull -q
%cd /content/revT2V
!pip install -q -e .
# no %autoreload: it breaks on Colab's Python 3.13 (removed `imp` module).
# After a git pull, restart the kernel and re-run from the top.

In [ ]:
import os
import torch

HF_TOKEN = "hf_..."  # paste your Hugging Face token here
os.environ["HF_TOKEN"] = HF_TOKEN
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
HF_REPO = "silentlooop/revt2v-ckpt"

PROMPT_SETS = {
    # fluids / gravity / walking: the original comparison set
    "physics": [
        "white smoke rising from a burning incense stick, black background",
        "red ink dropping into a clear glass of water, close-up",
        "orange juice being poured into a glass, close-up",
        "a red ball falling onto a wooden floor, static camera",
        "a man walking away from the camera down a long corridor, static camera",
    ]
}
PROMPT_SET = "physics"  # "physics" | "irreversible"
PROMPTS = PROMPT_SETS[PROMPT_SET]
OUT_DIR = f"/content/compare_results/{PROMPT_SET}"  # one folder per set, so runs don't overwrite
SEEDS = [0, 1]
GEN = dict(negative_prompt="watermark, text", num_inference_steps=25, guidance_scale=9.0,
           num_frames=16, height=256, width=256)
SHEET_FRAMES = [0, 5, 10, 15]
FPS = 8
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load the teacher once, then the trained adapters

In [ ]:
from revt2v.infer import MethodBank

bank = MethodBank(hf_repo_id=HF_REPO)
bank.pipeline.set_progress_bar_config(disable=True)

# attn_rotation: tries attn_rotation/latest.pt, then the original root latest.pt.
for method in ["attn_rotation", "conv_student"]:
    try:
        print(method, "loaded" if bank.load_adapter(method) else "-> no checkpoint, skipped")
    except Exception as exc:  # e.g. root checkpoint belongs to another method
        print(method, "-> skipped:", exc)

print("available methods:", bank.methods)

In [ ]:
def initial_noise(seed):
    g = torch.Generator(device="cuda").manual_seed(seed)
    return torch.randn((1, 4, GEN["num_frames"], GEN["height"] // 8, GEN["width"] // 8),
                       generator=g, device="cuda", dtype=bank.pipeline.unet.dtype)

def to_np(video):
    """(F, H, W, C) float tensor in [0, 1] -> float32 numpy."""
    return video.float().clamp(0, 1).cpu().numpy()

## 3. Exactness checks
(a) U-Net level: `unet_flipped(z, t, emb)` vs `flip(unet(flip(z), t, emb))`

In [ ]:
from contextlib import nullcontext
from revt2v.methods.conv_mirror import find_temporal_convs, flip_temporal_convs, set_flip_state
from revt2v.teacher import encode_prompt

unet = bank.pipeline.unet
print(len(find_temporal_convs(unet)), "temporal Conv3d layers")

z = initial_noise(0)
emb, _ = encode_prompt(bank.pipeline, PROMPTS[0])
emb = emb.to(unet.dtype)
t = torch.tensor([500], device="cuda")

adapters_off = bank.student.disable_adapter() if bank.student is not None else nullcontext()
try:
    with torch.no_grad(), adapters_off:
        mirrored = unet(z.flip(2), t, encoder_hidden_states=emb, return_dict=False)[0].flip(2)
        flip_temporal_convs(unet)
        flipped = unet(z, t, encoder_hidden_states=emb, return_dict=False)[0]
finally:
    set_flip_state(unet, ())

print(f"max |unet_flipped(z) - flip(unet(flip(z)))| = {(flipped.float() - mirrored.float()).abs().max().item():.3e}")

(b) Video level: `conv_oracle(latents=z)` vs `matched_target(prompt, z)`

In [ ]:
z = initial_noise(0)
oracle = to_np(bank.generate(PROMPTS[0], "conv_oracle", latents=z, **GEN)["video"])
target = to_np(bank.matched_target(PROMPTS[0], z, **GEN)["video"])
print(f"mean pixel diff (0-1 scale): {abs(oracle - target).mean():.4f}")

## 4. Metrics

In [ ]:
import cv2
import numpy as np

def flows(video):
    gray = [cv2.cvtColor((f * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY) for f in video]
    return np.stack([cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                     for a, b in zip(gray[:-1], gray[1:])])  # (F-1, H, W, 2)

def compare(video, target):
    fv, ft = flows(video), flows(target)
    mag_v, mag_t = np.linalg.norm(fv, axis=-1), np.linalg.norm(ft, axis=-1)
    cos = (fv * ft).sum(-1) / (mag_v * mag_t + 1e-6)
    weight = mag_t  # direction matters where the target actually moves
    return {
        "flow_cosine": float((cos * weight).sum() / (weight.sum() + 1e-6)),
        "motion_ratio": float(mag_v.mean() / (mag_t.mean() + 1e-6)),
        "pixel_diff": float(np.abs(video - target).mean()),
    }

## 5. Generate every method from the same noise

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display

COLUMNS = [m for m in ["teacher", "matched_target", "attn_rotation", "conv_oracle", "conv_student"]
           if m == "matched_target" or m in bank.methods]
rows, videos = [], {}

for p_idx, prompt in enumerate(PROMPTS):
    for seed in SEEDS:
        z = initial_noise(seed)
        out = {}
        for method in COLUMNS:
            if method == "matched_target":
                result = bank.matched_target(prompt, z, **GEN)
            else:
                result = bank.generate(prompt, method, latents=z, **GEN)
            out[method] = to_np(result["video"])
            # export_to_video expects float frames in [0, 1], not uint8
            export_to_video(list(out[method]), f"{OUT_DIR}/p{p_idx}_s{seed}_{method}.mp4", fps=FPS)
        videos[(p_idx, seed)] = out

        side_by_side = np.concatenate([out[m] for m in COLUMNS], axis=2)
        path = export_to_video(list(side_by_side), f"{OUT_DIR}/p{p_idx}_s{seed}_side_by_side.mp4", fps=FPS)
        print(f"[{p_idx}] seed={seed} {prompt}\n    " + " | ".join(COLUMNS))
        display(Video(path, embed=True, width=256 * len(COLUMNS)))

        for method in COLUMNS:
            if method != "matched_target":
                rows.append({"prompt": p_idx, "seed": seed, "method": method,
                             **compare(out[method], out["matched_target"])})

## 6. Summary table (vs matched target)

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df["frozen"] = df["motion_ratio"] < 0.3
df.to_csv(f"{OUT_DIR}/metrics.csv", index=False)

summary = df.groupby("method", sort=False).agg(
    flow_cosine=("flow_cosine", "mean"),
    motion_ratio=("motion_ratio", "mean"),
    pixel_diff=("pixel_diff", "mean"),
    frozen_runs=("frozen", "sum"),
    runs=("frozen", "size"),
)
summary.to_csv(f"{OUT_DIR}/summary.csv")
display(summary.round(3))
display(df.round(3))

## 7. Frame sheets (rows = methods, columns = frames)

In [ ]:
import matplotlib.pyplot as plt

for p_idx, prompt in enumerate(PROMPTS):
    out = videos[(p_idx, SEEDS[0])]
    fig, axes = plt.subplots(len(COLUMNS), len(SHEET_FRAMES),
                             figsize=(2.2 * len(SHEET_FRAMES), 2.2 * len(COLUMNS)), squeeze=False)
    for r, method in enumerate(COLUMNS):
        for c, f in enumerate(SHEET_FRAMES):
            ax = axes[r][c]
            ax.imshow(out[method][f])
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0: ax.set_title(f"frame {f}")
            if c == 0: ax.set_ylabel(method)
    fig.suptitle(f"[{p_idx}] {prompt} (seed {SEEDS[0]})", fontsize=9)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/p{p_idx}_frame_sheet.png", dpi=100)
    plt.show()

## 8. Zip and download

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(f"/content/compare_results_{PROMPT_SET}", "zip", OUT_DIR)
print(archive)
files.download(archive)